# Ames House Price Prediction

Predict `SalePrice` for homes in Ames, Iowa.

| No. | Title | Description |
| --- | --- | --- |
| 1 | Data Exploration and Cleaning | Load the dataset, perform exploratory analysis, handle missing values, and summarize data insights. |
| 2 | Feature Engineering and Selection | Create new features, transform variables, and select the most relevant features for modeling. |
| 3 | Model Building and Tuning | Implement and tune at least two regression models (e.g., Linear Regression, Random Forest) and compare their performance. |
| 4 | Evaluation and Submission | Generate predictions on the test set, evaluate results, and submit the final notebook and report. |


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("../data")
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print(train.shape, test.shape)
train.head()

## Milestone 1 — Data Exploration and Cleaning

Load the dataset, perform exploratory analysis, handle missing values, and summarize data insights.

In [ ]:
print(train.info())
print(train["SalePrice"].describe())
print(train.isna().sum().sort_values(ascending=False).head(15))

In [ ]:
for df in (train, test):
    df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
    df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
    df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])
    df.fillna(0, inplace=True)

print("Missing values left in train:", train.isna().sum().sum())

## Milestone 2 — Feature Engineering and Selection

Create new features, transform variables, and select the most relevant features for modeling.

In [ ]:
features = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt", "FullBath", "LotArea"]

for df in (train, test):
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

features = features + ["TotalSF", "HouseAge"]
X = train[features]
y = train["SalePrice"]
X_test = test[features]
X.head()

## Milestone 3 — Model Building and Tuning

Implement and tune at least two regression models (e.g., Linear Regression, Random Forest) and compare their performance.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42)

lr_score = -cross_val_score(lr, X, y, cv=5, scoring="neg_root_mean_squared_error").mean()
rf_score = -cross_val_score(rf, X, y, cv=5, scoring="neg_root_mean_squared_error").mean()

print("Linear Regression RMSE:", round(lr_score, 2))
print("Random Forest RMSE:", round(rf_score, 2))

## Milestone 4 — Evaluation and Submission

Generate predictions on the test set, evaluate results, and submit the final notebook and report.

In [ ]:
model = rf if rf_score < lr_score else lr
model.fit(X, y)
preds = model.predict(X_test)

submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("../submissions/submission.csv", index=False)
submission.head()